# Nome do Aluno: Milton Salgado Leandro
# Instituição de Ensino: Universidade Federal do Rio de Janeiro

# 1.1 — Preparação e Upload dos Dados

**Objetivo:** disponibilizar os cinco CSVs em um Volume do Unity Catalog, sem tratar os arquivos originais.

**Catálogo:** `workspace`  
**Volume:** `/Volumes/workspace/landing/inputs/`

**Arquivos:** `movies_info_TMDB_IMDB.csv`, `movies_financials_IMDB_TMDB.csv`, `movies_metrics_IMDB_TMDB.csv`, `credits_and_tags_IMDB_TMDB.csv` e `movies_reviews.csv`.

**Procedimento:** criar os Schemas `landing`, `bronze`, `silver` e `gold`, criar o Volume `landing.inputs` e fazer o upload pelo Catalog Explorer.


In [0]:
%sql
-- Limpar Schemas e Volumes para limpeza em caso de erros na ingestão inicial
USE CATALOG workspace;

TRUNCATE TABLE bronze.tb_movies_info;

TRUNCATE TABLE bronze.tb_movies_financials;

TRUNCATE TABLE bronze.tb_movies_metrics;

TRUNCATE TABLE bronze.tb_credits_and_tags;

TRUNCATE TABLE bronze.tb_movies_reviews;

TRUNCATE TABLE bronze.tb_cotacao_dolar;

In [0]:
%sql
-- Selecionar o catálogo do ambiente.
USE CATALOG workspace;

-- Criar os Schemas do projeto.
CREATE SCHEMA IF NOT EXISTS landing;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

-- Criar o Volume de arquivos de entrada.
CREATE VOLUME IF NOT EXISTS landing.inputs;

In [0]:
CATALOGO = "workspace"
BASE = f"/Volumes/{CATALOGO}/landing/inputs"

spark.sql(f"USE CATALOG {CATALOGO}")

arquivos_esperados = {
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv"
}

arquivos = dbutils.fs.ls(BASE)

display(arquivos)

arquivos_encontrados = {
    arquivo.name for arquivo in arquivos
}

faltantes = (
    arquivos_esperados - arquivos_encontrados
)

assert not faltantes, (
    f"Arquivos ausentes: {faltantes}"
)

print("Todos os 5 CSVs foram encontrados!")


path,name,size,modificationTime
dbfs:/Volumes/workspace/landing/inputs/credits_and_tags_IMDB_TMDB.csv,credits_and_tags_IMDB_TMDB.csv,22320678,1791335505000
dbfs:/Volumes/workspace/landing/inputs/movies_financials_IMDB_TMDB.csv,movies_financials_IMDB_TMDB.csv,1467743,1791335493000
dbfs:/Volumes/workspace/landing/inputs/movies_info_TMDB_IMDB.csv,movies_info_TMDB_IMDB.csv,33924063,1791335509000
dbfs:/Volumes/workspace/landing/inputs/movies_metrics_IMDB_TMDB.csv,movies_metrics_IMDB_TMDB.csv,3323884,1791335495000
dbfs:/Volumes/workspace/landing/inputs/movies_reviews.csv,movies_reviews.csv,1923431,1791335494000


Todos os 5 CSVs foram encontrados!


## 1.1.1 — Conferência dos CSVs

Conferir a contagem de linhas físicas, o total interpretado pelo Spark e a validade dos IDs. O parser usa `multiLine=False` porque há aspas malformadas na origem. A igualdade das contagens **não garante** recuperação de todas as colunas deslocadas; os CSVs originais permanecem no Volume.

In [0]:
from pyspark.sql import functions as F

CATALOGO = "workspace"
BASE = f"/Volumes/{CATALOGO}/landing/inputs"

arquivos = {
    "movies_info_TMDB_IMDB.csv":
        "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv":
        "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv":
        "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv":
        "tb_credits_and_tags",
    "movies_reviews.csv":
        "tb_movies_reviews"
}

for arquivo, tabela in arquivos.items():

    caminho = f"{BASE}/{arquivo}"

    # Contar linhas físicas, descontando o cabeçalho.
    linhas = spark.read.text(caminho).count() - 1

    # Validar o parser linha a linha antes da ingestão.
    df_teste = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", False)
        .option("mode", "PERMISSIVE")
        .csv(caminho)
    )

    total = df_teste.count()

    ids_invalidos = df_teste.filter(
        ~F.coalesce(
            F.col("id").rlike(r"^[0-9]+$"),
            F.lit(False)
        )
    ).count()

    print(f"\n{tabela}")
    print(f"Linhas esperadas: {linhas}")
    print(f"Linhas lidas: {total}")
    print(f"IDs inválidos: {ids_invalidos}")

    assert total == linhas, (
        f"Divergência em {arquivo}"
    )

    assert ids_invalidos == 0, (
        f"IDs inválidos em {arquivo}"
    )



tb_movies_info
Linhas esperadas: 106930
Linhas lidas: 106930
IDs inválidos: 0

tb_movies_financials
Linhas esperadas: 106165
Linhas lidas: 106165
IDs inválidos: 0

tb_movies_metrics
Linhas esperadas: 107364
Linhas lidas: 107364
IDs inválidos: 0

tb_credits_and_tags
Linhas esperadas: 106320
Linhas lidas: 106320
IDs inválidos: 0

tb_movies_reviews
Linhas esperadas: 32412
Linhas lidas: 32412
IDs inválidos: 0


# 1.2 — Ingestão dos CSVs na Bronze

**Origem:** cinco CSVs no Volume `landing.inputs`.  
**Destino:** `bronze.tb_movies_info`, `tb_movies_financials`, `tb_movies_metrics`, `tb_credits_and_tags` e `tb_movies_reviews`.

**Regras:** preservar nomes e valores lidos como texto (`inferSchema=False`), registrar `ingestion_datetime` e salvar em Delta com **Append**, sem limpeza ou deduplicação. Anomalias de formato são tratadas na Silver, mantendo os arquivos originais disponíveis para auditoria.

**Reexecução:** o `Append` adiciona uma nova carga dos mesmos CSVs. Não repetir a ingestão por engano. Para refazer a carga inicial, limpar **somente as tabelas deste projeto**, de maneira controlada, antes de rodar esta seção.


In [0]:
from pyspark.sql import functions as F

# Ambiente e mapeamento das fontes
CATALOGO = "workspace"
BASE = f"/Volumes/{CATALOGO}/landing/inputs"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# Mapeamento dos arquivos para as tabelas
arquivos = {
    "movies_info_TMDB_IMDB.csv":
        "tb_movies_info",

    "movies_financials_IMDB_TMDB.csv":
        "tb_movies_financials",

    "movies_metrics_IMDB_TMDB.csv":
        "tb_movies_metrics",

    "credits_and_tags_IMDB_TMDB.csv":
        "tb_credits_and_tags",

    "movies_reviews.csv":
        "tb_movies_reviews"
}

# Ingestão dos dados brutos
for arquivo, tabela in arquivos.items():

    caminho = f"{BASE}/{arquivo}"

    print(f"Ingerindo: {arquivo}")

    # Preservar os valores como texto.
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", False)
        .option("mode", "PERMISSIVE")
        .csv(caminho)
    )

    # Timestamp do momento da ingestão.
    df_bronze = df.withColumn(
        "ingestion_datetime",
        F.current_timestamp()
    )

    # Armazenamento em Delta com Append.
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"bronze.{tabela}")
    )

    print(f"Carga concluída: bronze.{tabela}")

print("Ingestão dos CSVs concluída!")


Ingerindo: movies_info_TMDB_IMDB.csv
Carga concluída: bronze.tb_movies_info
Ingerindo: movies_financials_IMDB_TMDB.csv
Carga concluída: bronze.tb_movies_financials
Ingerindo: movies_metrics_IMDB_TMDB.csv
Carga concluída: bronze.tb_movies_metrics
Ingerindo: credits_and_tags_IMDB_TMDB.csv
Carga concluída: bronze.tb_credits_and_tags
Ingerindo: movies_reviews.csv
Carga concluída: bronze.tb_movies_reviews
Ingestão dos CSVs concluída!


## 1.2.1 — Validação das Tabelas Bronze

Conferir contagem, esquema original e presença de `ingestion_datetime` nas cinco tabelas. A ingestão deve preservar uma linha por registro físico da origem.


In [0]:
tabelas_bronze = list(arquivos.values())

for tabela in tabelas_bronze:

    df = spark.table(f"bronze.{tabela}")

    print(f"\nTABELA: {tabela}")
    print(f"Registros: {df.count()}")

    # Verificação da coluna de ingestão
    assert "ingestion_datetime" in df.columns

    df.printSchema()

    display(df.limit(5))


TABELA: tb_movies_info
Registros: 106930
root
 |-- id: string (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T03:39:41.896Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T03:39:41.896Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T03:39:41.896Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T03:39:41.896Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T03:39:41.896Z



TABELA: tb_movies_financials
Registros: 106165
root
 |-- id: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-10-07T03:39:45.220Z
299536,300000000,2052415039,2026-10-07T03:39:45.220Z
299534,356000000,2800000000,2026-10-07T03:39:45.220Z
475557,55000000,1074458282,2026-10-07T03:39:45.220Z
271110,250000000,Não Informado,2026-10-07T03:39:45.220Z



TABELA: tb_movies_metrics
Registros: 107364
root
 |-- id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-07T03:39:47.426Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-07T03:39:47.426Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-07T03:39:47.426Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-07T03:39:47.426Z
271110,70.741,7.4,21541,7.8,947222,2026-10-07T03:39:47.426Z



TABELA: tb_credits_and_tags
Registros: 106320
root
 |-- id: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-07T03:39:49.612Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-07T03:39:49.612Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-07T03:39:49.612Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-07T03:39:49.612Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-07T03:39:49.612Z



TABELA: tb_movies_reviews
Registros: 32412
root
 |-- id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: string (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-07T03:39:52.241Z
637007,Lucas Reis 602,3.9,null,2026-10-07T03:39:52.241Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-07T03:39:52.241Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-07T03:39:52.241Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-07T03:39:52.241Z


# 1.2.2 — Cotação do Dólar via API PTAX

**Origem:** API `CotacaoDolarPeriodo` do Banco Central.  
**Destino:** `bronze.tb_cotacao_dolar`.

Usar widgets para datas inicial e final (`MM-DD-AAAA`), com consulta padrão aos últimos sete dias. Armazenar `dataHoraCotacao`, `cotacaoCompra` e `ingestion_datetime` em Delta/Append. Dias sem cotação (feriados e fins de semana) serão preenchidos **somente na Silver**.


In [0]:
import requests
from datetime import date, timedelta
from pyspark.sql import functions as F

# Widgets de parametrização.
dbutils.widgets.text(
    "data_inicio",
    (date.today() - timedelta(days=7))
    .strftime("%m-%d-%Y")
)

dbutils.widgets.text(
    "data_fim",
    date.today().strftime("%m-%d-%Y")
)

inicio = dbutils.widgets.get("data_inicio")
fim = dbutils.widgets.get("data_fim")

# Endpoint oficial PTAX.
url = (
    "https://olinda.bcb.gov.br/olinda/"
    "servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo("
    "dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)"
)

params = {
    "@dataInicial": f"'{inicio}'",
    "@dataFinalCotacao": f"'{fim}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}

# Requisição à API.
resposta = requests.get(
    url, params=params, timeout=30
)

resposta.raise_for_status()

dados = resposta.json().get("value", [])

if not dados:
    raise ValueError(
        "Nenhuma cotação encontrada no período."
    )

# Criar DataFrame mantendo os campos da API.
df_cotacao = spark.createDataFrame(dados).select(
    "dataHoraCotacao",
    "cotacaoCompra"
)

# Registrar o momento da ingestão.
df_cotacao = df_cotacao.withColumn(
    "ingestion_datetime",
    F.current_timestamp()
)

# Salvar em Delta, no modo Append.
(
    df_cotacao.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.tb_cotacao_dolar")
)

display(df_cotacao)

dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-30 13:11:44.783262,5.1803,2026-10-07T03:40:03.489Z
2026-10-01 13:10:35.4469,5.2073,2026-10-07T03:40:03.489Z
2026-10-02 13:03:16.256632,5.2232,2026-10-07T03:40:03.489Z
2026-10-05 13:07:36.558602,4.9853,2026-10-07T03:40:03.489Z
2026-10-06 13:03:21.267287,4.9692,2026-10-07T03:40:03.489Z


## 1.2.3 — Validação da API

Conferir os campos `dataHoraCotacao`, `cotacaoCompra` e `ingestion_datetime`, além dos valores retornados pela API. A PTAX pode não oferecer cotações em todos os dias consultados.


In [0]:

df = spark.table("bronze.tb_cotacao_dolar")

print("Total de cotações:", df.count())

df.printSchema()

display(
    df.orderBy(
        F.col("dataHoraCotacao").desc()
    )
)

Total de cotações: 5
root
 |-- dataHoraCotacao: string (nullable = true)
 |-- cotacaoCompra: double (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-10-06 13:03:21.267287,4.9692,2026-10-07T03:40:02.553Z
2026-10-05 13:07:36.558602,4.9853,2026-10-07T03:40:02.553Z
2026-10-02 13:03:16.256632,5.2232,2026-10-07T03:40:02.553Z
2026-10-01 13:10:35.4469,5.2073,2026-10-07T03:40:02.553Z
2026-09-30 13:11:44.783262,5.1803,2026-10-07T03:40:02.553Z


## 1.2.4 — Conferência Final da Bronze

Comparar o número de registros armazenados em cada tabela com a quantidade de linhas físicas de seu CSV (excluído o cabeçalho). **Não** reexecutar a ingestão apenas para validar: em `Append`, isso duplicaria a carga.

In [0]:
# Validar as quantidades após a carga.
for arquivo, tabela in arquivos.items():

    caminho = f"{BASE}/{arquivo}"

    total_original = (
        spark.read.text(caminho).count() - 1
    )

    total_bronze = (
        spark.table(f"bronze.{tabela}")
        .count()
    )

    print(
        f"{tabela}: "
        f"{total_bronze}/{total_original}"
    )

    assert total_bronze == total_original

print("Todas as contagens conferem!")

tb_movies_info: 106930/106930
tb_movies_financials: 106165/106165
tb_movies_metrics: 107364/107364
tb_credits_and_tags: 106320/106320
tb_movies_reviews: 32412/32412
Todas as contagens conferem!


In [0]:
%sql
SELECT 'Informações' AS tabela, COUNT(*) AS total
FROM bronze.tb_movies_info

UNION ALL

SELECT 'Financeiro', COUNT(*)
FROM bronze.tb_movies_financials

UNION ALL

SELECT 'Métricas', COUNT(*)
FROM bronze.tb_movies_metrics

UNION ALL

SELECT 'Créditos', COUNT(*)
FROM bronze.tb_credits_and_tags

UNION ALL

SELECT 'Avaliações', COUNT(*)
FROM bronze.tb_movies_reviews

UNION ALL

SELECT 'Cotação', COUNT(*)
FROM bronze.tb_cotacao_dolar;

tabela,total
Informações,106930
Financeiro,106165
Métricas,107364
Créditos,106320
Avaliações,32412
Cotação,5
